# SBOM-to-Audit — Stage 5.5.1 Colab checkpoint

This checkpoint independently verifies the authoritative historical EPSS gate for CVE-2024-3400, the complete repository release gate, and the EPSS ablation. It uses an isolated Python environment and preserves the raw FIRST API response, pinned daily archive, extracted row and verification report in the downloadable evidence bundle.


In [ ]:
REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = "main"  # Replace with a Stage 5.5.1 tag after one exists.
print(REPO_URL, REF)


In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path
WORKDIR = Path("/content/sbom_to_audit")
VENV = Path("/content/sbom_to_audit_stage551_venv")
for path in (WORKDIR, VENV):
    if path.exists():
        shutil.rmtree(path)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Tested Git commit:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
try:
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
print("PASS: isolated dependency integrity")


In [ ]:
# Download and compare the authoritative date-specific FIRST API record and pinned official archive.
EPSS_DIR = Path("/content/stage551_epss_authoritative_evidence")
EPSS_REPORT = EPSS_DIR / "historical_epss_verification.json"
if EPSS_DIR.exists():
    shutil.rmtree(EPSS_DIR)
subprocess.run([
    str(PYTHON), "scripts/verify_historical_epss.py", "--online",
    "--output-dir", str(EPSS_DIR), "--report", str(EPSS_REPORT)
], check=True)
import json
verification = json.loads(EPSS_REPORT.read_text())
assert verification["status"] == "authoritative_dual_source_verified"
assert all(verification["checks"].values())
assert verification["api_record"]["epss"] == "0.95732"
assert verification["archive_record"]["percentile"] == "0.99721"
assert verification["archive_record"]["model_version"] == "v2023.03.01"
assert (EPSS_DIR / "epss_scores-2024-04-15.csv.gz").is_file()
print("PASS: authoritative dual-source historical EPSS verification")


In [ ]:
# Execute the canonical release gate after authoritative verification.
RELEASE_REPORT = Path("/content/stage551_colab_release_validation.json")
subprocess.run([str(PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)], check=True)
release = json.loads(RELEASE_REPORT.read_text())
assert release["status"] == "PASS", release["errors"]
assert len(release["deterministic_hashes"]) == 53
print("PASS: release gate and 53 deterministic outputs")


In [ ]:
# Generate a separate historical replay, reference EvidencePack and verified paper assets.
OUTPUTS = Path("/content/stage551_checkpoint_outputs")
ASSETS = Path("/content/stage551_checkpoint_paper_assets")
for path in (OUTPUTS, ASSETS):
    if path.exists():
        shutil.rmtree(path)
subprocess.run([str(PYTHON), "-m", "sbom_to_audit.cli", "--scenario", "data/scenarios/historical_cve_2024_3400_reference.yaml", "--output-root", str(OUTPUTS)], check=True)
subprocess.run([str(PYTHON), "scripts/run_historical_replay.py", "--output-root", str(OUTPUTS), "--epss-verification-report", str(EPSS_REPORT)], check=True)
subprocess.run([str(PYTHON), "paper_assets/scripts/build_stage55_assets.py", "--output-root", str(OUTPUTS), "--destination", str(ASSETS)], check=True)
subprocess.run([str(PYTHON), "paper_assets/scripts/build_stage551_assets.py", "--output-root", str(OUTPUTS), "--destination", str(ASSETS)], check=True)


In [ ]:
bundle = json.loads((OUTPUTS / "historical_public/cve_2024_3400_public_bundle.json").read_text())
ablation = json.loads((OUTPUTS / "historical_public/cve_2024_3400_epss_ablation.json").read_text())
assert bundle["provisional_source_ids"] == []
assert bundle["manuscript_eligibility"] is True
assert bundle["historical_epss_verification"]["status"] == "authoritative_dual_source_verified"
assert ablation["state_trajectory_changed"] is False
assert ablation["final_E_t_with_epss"] == ablation["final_E_t_without_epss"] == 1.0
print("PASS: eligibility blocker removed")
print("PASS: EPSS ablation leaves state trajectory unchanged")


In [ ]:
# Preserve the checkpoint and raw authoritative evidence.
import datetime as dt, hashlib, zipfile
environment = {
    "checked_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repository": REPO_URL,
    "reference": REF,
    "git_commit": COMMIT,
    "kernel_python": sys.version,
    "isolated_python": subprocess.check_output([str(PYTHON), "--version"], text=True).strip(),
    "platform": platform.platform(),
    "release_status": release["status"],
    "historical_epss_status": verification["status"],
}
ENV = Path("/content/stage551_colab_environment.json")
ENV.write_text(json.dumps(environment, indent=2) + "\n")
BUNDLE = Path("/content/stage551_colab_checkpoint_evidence.zip")
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RELEASE_REPORT, RELEASE_REPORT.name)
    archive.write(ENV, ENV.name)
    for root in (EPSS_DIR, OUTPUTS, ASSETS):
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, f"{root.name}/{path.relative_to(root)}")
digest = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
print("Created:", BUNDLE)
print("Tested Git commit:", COMMIT)
print("Colab evidence-bundle SHA-256:", digest)
print("Download the ZIP from Colab's Files panel and preserve it unchanged.")
